# RDD Caching and Persistence Strategies

**Module**: 02_rdd  
**Notebook**: 05_rdd_caching_and_persistence.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How RDD caching and persistence work, comparing storage levels (MEMORY_ONLY, MEMORY_AND_DISK, MEMORY_ONLY_SER), and cache eviction.
- **Why does this topic exist?**: Caching prevents redundant recomputation across multiple actions, turning 10-minute pipelines into 10-second queries.
- **What problem does it solve?**: Blind caching wastes executor RAM, causes memory spill to disk, and chokes shuffle execution memory.
- **Where is it used in real-world Data Engineering?**: Iterative machine learning algorithms (K-Means, Gradient Descent) and branching ETL pipelines.

---

## 2. Storage Levels Comparison

| Storage Level | Space Used | CPU Time | In Memory | On Disk | Serialized | Off-Heap |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **MEMORY_ONLY** | High | Low | Yes | No | No | No |
| **MEMORY_ONLY_SER** | Low | High | Yes | No | Yes | No |
| **MEMORY_AND_DISK** | High | Medium | Some | Some | No | No |
| **MEMORY_AND_DISK_SER** | Low | High | Some | Some | Yes | No |
| **DISK_ONLY** | Low | High | No | Yes | Yes | No |

- `rdd.cache()` is an alias for `rdd.persist(StorageLevel.MEMORY_ONLY)`.
- Always call `rdd.unpersist()` when finished with the cached dataset.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("RDD_Caching_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Hands-On: Caching an Expensive RDD Computation

In [ ]:
from pyspark import StorageLevel

sc = spark.sparkContext
orders_rdd = sc.textFile("/opt/spark-data/retail/orders.csv")

# Perform an expensive parsing step
parsed_orders = (
    orders_rdd
    .filter(lambda l: not l.startswith("order_id"))
    .map(lambda l: l.split(","))
    .map(lambda c: (c[7], float(c[5]) * int(c[4]))) # (status, total_price)
)

# Persist in memory
parsed_orders.persist(StorageLevel.MEMORY_ONLY)
print(f"Is RDD cached?: {parsed_orders.is_cached}")
print(f"Storage Level:  {parsed_orders.getStorageLevel()}")

## 4. Benchmarking Cached vs Uncached Performance

Watch the execution time drop dramatically on the second Action!

In [ ]:
# Action 1: First time evaluates lineage AND caches into BlockManager
t0 = time.time()
count_1 = parsed_orders.count()
t_action1 = time.time() - t0

# Action 2: Evaluates directly from in-memory BlockManager!
t0 = time.time()
count_2 = parsed_orders.count()
t_action2 = time.time() - t0

print(f"Action 1 (Lineage + Cache populate): {t_action1:.4f}s")
print(f"Action 2 (Directly from RAM cache):  {t_action2:.4f}s")
print(f"Speedup from caching:                 {t_action1 / t_action2:.2f}x")

## 5. Unpersisting to Free Memory

Always unpersist when your branching pipeline finishes!

In [ ]:
parsed_orders.unpersist()
print(f"Is RDD cached after unpersist?: {parsed_orders.is_cached}")

---

## Experiment: Inspecting Cache in Spark UI Storage Tab

- **1. What are we testing?**: Persist an RDD, trigger an action, and verify that cached partitions appear in the Spark UI Storage tab.
- **2. What do we expect?**: Storage tab shows memory used and percentage cached.

In [ ]:
rdd_sample = sc.parallelize(range(100000), 4).map(lambda x: x * 3)
rdd_sample.persist(StorageLevel.MEMORY_ONLY)
# Action to populate cache
rdd_sample.count()

print("Check http://localhost:4040/storage/ to see the cached RDD blocks!")
print(f"Storage level: {rdd_sample.getStorageLevel()}")
rdd_sample.unpersist()

**3. What happened?**: Cached blocks were populated in the BlockManager and visible in Spark UI.  
**4. Why did it happen?**: Calling an action materializes the RDD partitions and stores them in the Storage Memory pool.  
**5. Practical Engineering Lesson**: Always monitor the Storage tab in Spark UI to ensure cached datasets fit in RAM without spilling.

---

## Practical Exercises

### Exercise 1 (Beginner): Persist with MEMORY_AND_DISK
**Problem Statement**: Persist parsed_orders using StorageLevel.MEMORY_AND_DISK, run count(), and verify getStorageLevel().  
**Input / Context**: StorageLevel.MEMORY_AND_DISK

In [ ]:
# Exercise 1: MEMORY_AND_DISK persistence

### Exercise 2 (Intermediate): Compare Cached vs Uncached Branching Aggregations
**Problem Statement**: Execute sum() and max() on an unpersisted RDD vs cached RDD and compare total execution time.  
**Input / Context**: sc.parallelize(range(1000000), 4).map(lambda x: x**2)

In [ ]:
# Exercise 2: Branching actions benchmark

### Exercise 3 (Advanced): Inspect Serialized Storage Level Memory Footprint
**Problem Statement**: Persist an RDD with MEMORY_ONLY and MEMORY_ONLY_SER and compare behavior.  
**Input / Context**: StorageLevel.MEMORY_ONLY_SER

In [ ]:
# Exercise 3: Serialized storage

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Persist with MEMORY_AND_DISK):
```python
p = sc.parallelize(range(100)).persist(StorageLevel.MEMORY_AND_DISK)
p.count()
print("Storage level:", p.getStorageLevel())
p.unpersist()
```

#### Solution 2 (Compare Cached vs Uncached Branching Aggregations):
```python
r = sc.parallelize(range(500000), 4).map(lambda x: x * 2).persist()
# Warm up cache with action
r.count()
t0 = time.time()
s_val = r.sum()
m_val = r.max()
print(f"Sum: {s_val}, Max: {m_val} executed in {time.time()-t0:.4f}s from cache")
r.unpersist()
```

#### Solution 3 (Inspect Serialized Storage Level Memory Footprint):
```python
r_ser = sc.parallelize([f"Str_{i}"for i in range(50000)]).persist(StorageLevel.MEMORY_ONLY_SER)
r_ser.count()
print("Serialized level:", r_ser.getStorageLevel())
r_ser.unpersist()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **When to Cache**: Cache only when an expensive RDD is reused across 2 or more actions.
- **unpersist() Discipline**: Always release memory when finished to avoid choking execution pools.
- **Storage Levels**: MEMORY_ONLY is fastest; MEMORY_AND_DISK prevents recomputation if RAM is full.

### Essential Technical Terms
- `cache()`: Persist with default MEMORY_ONLY (RDD) or MEMORY_AND_DISK (DataFrame).
- `StorageLevel`: Configuration defining memory, disk, serialization, and replication flags.
- `unpersist()`: Eagerly removes cached blocks from the BlockManager.

### Common Engineering Mistakes
- **Caching Single-Use RDDs**: Adds serialization and BlockManager overhead with zero reuse benefit.
- **Forgetting to Unpersist**: Leads to memory eviction and disk spilling in subsequent jobs.

### Interview Questions on This Topic
- **Q: What is the difference between cache() and persist()?**
  - *A*: cache() uses the default storage level; persist() accepts a custom StorageLevel parameter.
- **Q: Why might caching make a job slower?**
  - *A*: If the data is only used once, caching incurs storage overhead without any recomputation savings.

### What to Study Next
Module 02 complete! Proceed to Module 03: DataFrames and Spark SQL.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')